# Utiliser et adapter le modèle

[English](adapt_the_model.ipynb) | **Français**

Ce notebook montre, en six petites étapes, comment utiliser ce modèle pour votre propre question : de la reproduction des résultats jusqu'au remplacement d'une loi de vieillissement. Les explications sont dans [`docs/GUIDE.fr.md`](../docs/GUIDE.fr.md).

Lancez les cellules dans l'ordre (Maj + Entrée). Le notebook fonctionne depuis le dossier `examples` du dépôt.

## Installation

In [ ]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"           # le paquet h2ageing de ce dépôt
sys.path.insert(0, str(SRC))

import numpy as np
from h2ageing import load_config, with_values, run

cfg = load_config()                         # toutes les entrées, depuis config/default.yaml

## Niveau 1 : reproduire les résultats

In [ ]:
ref = run(cfg)
ref["chain"][["case", "total_g_per_km", "rise_pct"]].round(1)

## Niveau 2 : poser une question au modèle

*Et si le stack de pile dure 5 000 h ?* `with_values` renvoie une copie modifiée de la configuration.

In [ ]:
r = run(with_values(cfg, **{"cases.central.fuel_cell.hours_to_end_of_life": 5000.0}))
r["chain"][["case", "total_g_per_km", "rise_pct"]].round(1)

*À partir de quelle durée de vie un seul stack suffit-il ?*

In [ ]:
for d in [3000, 4100, 5000, 5554, 6000, 7000]:
    x = run(with_values(cfg, **{"cases.central.fuel_cell.hours_to_end_of_life": float(d)}))
    stacks = x["life"].set_index("case").loc["central", "fuel_cell_stacks"]
    rise = x["chain"].set_index("case").loc["central", "rise_pct"]
    print(f"{d} h -> {stacks:.0f} stack(s), chain rise +{rise:.1f} %")

En dessous de 5 554 h (les heures de fonctionnement du véhicule), deux stacks sont nécessaires et le résultat bouge à peine ; au-dessus, un seul stack suffit.

## Niveau 3 : votre propre cas

Un constructeur dont le stack dure 7 000 h et dont l'électrolyseur fonctionne à l'électricité éolienne. Plusieurs valeurs peuvent être changées à la fois ; la même chose peut être écrite dans une copie de `config/default.yaml` et lancée avec `python scripts/run.py --config config/mon_cas.yaml`.

In [ ]:
my_case = with_values(cfg, **{
    "cases.central.fuel_cell.hours_to_end_of_life": 7000.0,   # notre propre essai d'endurance
    "usage.electricity_electrolyser": "wind_FR",              # contrat d'achat d'électricité éolienne
})
run(my_case)["chain"][["case", "total_g_per_km", "rise_pct"]].round(1)

## Niveau 4 : vos propres constantes de vieillissement

Une perte de tension de 27 mV à 1,8 A/cm² après 1 000 h, mesurée sur une autre cellule, au lieu de 45 mV dans le cas pessimiste.

In [ ]:
r = run(with_values(cfg, **{"cases.pessimistic.fuel_cell.voltage_loss_at_1p8_A_cm2_1000h_V": 0.027}))
r["life"][["case", "fuel_cell_end_of_life_h", "fuel_cell_stacks"]].round(0)

## Niveau 5 : une autre loi de vieillissement

Une résistance de la pile qui augmente comme la racine carrée du temps, avec la même valeur que la loi linéaire à 1 000 h. Seule la loi est écrite ; la fin de vie, les stacks et l'ACV sont recalculés.

In [ ]:
from h2ageing import pemfc

original_law = pemfc.FuelCell.resistance

def my_resistance(self, t, p):
    t = np.asarray(t, float)
    return self.R0 * (1 + p.get("k_R", 0.0) * 1000 * np.sqrt(t / 1000))

pemfc.FuelCell.resistance = my_resistance
r = run(cfg)
pemfc.FuelCell.resistance = original_law        # remettre la loi d'origine

r["life"][["case", "fuel_cell_end_of_life_h", "fuel_cell_stacks"]].round(0)

In [ ]:
r["chain"][["case", "total_g_per_km", "rise_pct"]].round(1)

## Niveau 6 : vos propres données

Les mesures, les inventaires et les impacts sont de simples fichiers dans `data/`. Remplacez-les, puis réajustez ou reconstruisez comme expliqué dans [`docs/GUIDE.fr.md`](../docs/GUIDE.fr.md) (section 3.5).

## Vérification

Les résultats de référence ne changent pas tant que la configuration par défaut et les lois d'origine sont utilisées :

In [ ]:
assert abs(run(cfg)["chain"].set_index("case").loc["central", "rise_pct"] - 31.07) < 0.01
print("résultats de référence inchangés")